# Production per-event timing breakdown

Mirrors the methodology of `computational_performance_evaluation.ipynb`,
but for the **production data-mode pipeline** used by `lucid-run-job`.
Each step from the per-event loop in `lucid/sources/event_io.py:952-1357`
is benchmarked **in isolation**, with warmup + `block_until_ready()` for
JAX outputs, so timings reflect the actual cost of each step rather
than where Python happens to demand the bytes.

Setup matches `lucid/production/run_job.py:206-215`:
`K=12, is_data=True, temperature=0.0, hit_mode='realistic'`,
`apply_smearing=False` at simulator level (smearing applied separately
below — same as production).

Inputs: a PhotonSim ROOT file from `dataprod_01_mu.json`. Set the
`ROOT_FILE` path in the config cell.


In [ ]:
! pip install -e ../

In [ ]:
import sys, time, os
sys.path.append('..')
import numpy as np
import jax
import jax.numpy as jnp
import uproot

from lucid.simulation import setup_event_simulator
from lucid.detector_params import ParticleParams
from lucid.sources.event_io import (
    read_particle_data_from_photonsim,
    build_interaction_metadata,
    _compute_contained,
    smear_charges_SK_like,
    smear_times,
)

print('jax devices:', jax.devices())

In [ ]:
# === Configuration ===

# Point at a PhotonSim ROOT file produced by dataprod_01_mu.json. Easiest
# route: run `lucid-run-job --config dataprod_01_mu.json --output-dir <out>
# --job-id 1 --n-events 20` once and use the resulting `output_job_*.root`.
ROOT_FILE = os.environ.get('PHOTONSIM_ROOT', '/out/output_job_000001.root')

DETECTOR_JSON = '../config/SK_geom_config.json'
PHYSICS_JSON  = '../config/SK_physics_config.json'

# Production settings.
K = 12

# Bench knobs — keep WARMUP small to avoid waiting forever when stages
# trigger JIT recompiles on first call.
WARMUP = 3
RUNS   = 10

EVENT_IDX = 0  # which event in the ROOT file to use for the per-step bench

In [ ]:
# === PAD_SIZE scan === — same logic as event_io.py:879-884.
with uproot.open(ROOT_FILE) as f:
    tree = f['OpticalEvents']
    photon_counts = np.array(
        [len(a) for a in tree['Photon_OriginX'].array(library='np')])
n_events_in_file = len(photon_counts)
PAD_SIZE = int(photon_counts.max()) + 1

print(f'n_events:         {n_events_in_file}')
print(f'photon counts:    min={photon_counts.min():,}  '
      f'mean={photon_counts.mean():,.0f}  '
      f'max={photon_counts.max():,}')
print(f'PAD_SIZE:         {PAD_SIZE:,}  (max + 1, fixed for the whole batch)')
print(f'avg padding ratio (PAD_SIZE / actual): '
      f'{PAD_SIZE / max(photon_counts.mean(), 1):.2f}x  '
      '— amount of wasted kernel work per event on average')

In [ ]:
# === Simulator setup (Python only — no JIT yet) ===
t0 = time.perf_counter()
event_simulator = setup_event_simulator(
    DETECTOR_JSON, 0, K=K,
    is_data=True, temperature=0.0,
    apply_smearing=False,  # production applies it separately below
    physics_config=PHYSICS_JSON,
    default_detector_params=True,
)
setup_s = time.perf_counter() - t0
print(f'setup_event_simulator (no JIT yet): {setup_s:.3f}s')

In [ ]:
# === Benchmark helpers ===
def _block(out):
    """block_until_ready every leaf of a pytree that supports it."""
    jax.tree.map(
        lambda x: x.block_until_ready() if hasattr(x, 'block_until_ready') else x,
        out)

def bench(fn, *args, warmup=WARMUP, runs=RUNS, block=False):
    """Run fn(*args) repeatedly; return (last_output, times_array_s)."""
    for _ in range(warmup):
        out = fn(*args)
        if block:
            _block(out)
    times = []
    for _ in range(runs):
        s = time.perf_counter()
        out = fn(*args)
        if block:
            _block(out)
        times.append(time.perf_counter() - s)
    return out, np.array(times)

def fmt(times):
    a = 1000.0 * times
    return (f'mean={a.mean():7.2f} ms   std={a.std():6.2f} ms   '
            f'min={a.min():7.2f}   max={a.max():7.2f}')

In [ ]:
# === Step 1 — read_particle_data_from_photonsim ===
def step_root_read():
    return read_particle_data_from_photonsim(
        ROOT_FILE, EVENT_IDX,
        include_track_segments=False,
        include_segment_index=False,
    )

particle_data, t_root = bench(step_root_read)
n_particles = particle_data['n_particles']
total_photons = len(particle_data['photon_origins'])
print(f'event {EVENT_IDX}: n_particles={n_particles}  total_photons={total_photons:,}')
print(f'root_read     : {fmt(t_root)}')

In [ ]:
# === Step 2 — NumPy preprocessing (scatter/pad to (n_particles, PAD_SIZE, ...)) ===
default_direction = np.array([0.0, 0.0, 1.0], dtype=np.float32)

def step_preprocess():
    particles = particle_data['particles']
    n_p = particle_data['n_particles']
    all_o = particle_data['photon_origins'].astype(np.float32, copy=False)
    all_d = particle_data['photon_directions'].astype(np.float32, copy=False)
    all_t = particle_data['photon_times'].astype(np.float32, copy=False)
    all_w = particle_data['photon_wavelengths'].astype(np.float32, copy=False)

    bo = np.zeros((n_p, PAD_SIZE, 3), dtype=np.float32)
    bd = np.tile(default_direction, (n_p, PAD_SIZE, 1))
    bt = np.zeros((n_p, PAD_SIZE), dtype=np.float32)
    bw = np.zeros((n_p, PAD_SIZE), dtype=np.float32)
    Np = np.zeros(n_p, dtype=np.int32)
    te = np.zeros(n_p, dtype=np.float32)
    tp = np.zeros((n_p, 3), dtype=np.float32)
    td = np.zeros((n_p, 3), dtype=np.float32)

    for i, p in enumerate(particles):
        idx = p['photon_indices']
        N = len(idx)
        Np[i] = N
        ti = p['track_info']
        if ti is not None:
            te[i] = ti['energy']
            tp[i] = ti['position']
            td[i] = ti['direction']
        else:
            te[i] = particle_data['primary_energy']
            td[i] = [0.0, 0.0, 1.0]
        if N > 0:
            bo[i, :N] = all_o[idx]
            bd[i, :N] = all_d[idx]
            bt[i, :N] = all_t[idx]
            bw[i, :N] = all_w[idx]
    return bo, bd, bt, bw, Np, te, tp, td

pre_out, t_pre = bench(step_preprocess)
bo, bd, bt, bw, Np, te, tp, td = pre_out
print(f'preprocess    : {fmt(t_pre)}')
print(f'  shapes: bo={bo.shape}  PAD_SIZE={PAD_SIZE:,}  per-particle Np={Np.tolist()}')

In [ ]:
# === Step 3 — host -> device (jax.device_put) ===
def step_device_put():
    return (
        jax.device_put(bo), jax.device_put(bd), jax.device_put(bt), jax.device_put(bw),
        jax.device_put(Np), jax.device_put(te), jax.device_put(tp), jax.device_put(td),
    )

arrays_d, t_dput = bench(step_device_put, block=True)
bo_d, bd_d, bt_d, bw_d, Np_d, te_d, tp_d, td_d = arrays_d
print(f'device_put    : {fmt(t_dput)}  (8 arrays; total {bo.nbytes + bd.nbytes + bt.nbytes + bw.nbytes:,} bytes)')

In [ ]:
# === Step 4 — kernel-only timing (vmap over particles + block_until_ready) ===
def simulate_single_particle(track_E, track_pos, track_dir,
                             ph_o, ph_d, ph_t, ph_w, N, key):
    track_params = ParticleParams.from_cartesian(
        energy=track_E, position=track_pos, direction=track_dir, t0=0.0)
    photonsim_data = {
        'photon_origins': ph_o,
        'photon_directions': ph_d,
        'photon_times': ph_t,
        'wavelengths': ph_w,
        'N': N,
        'apply_rotation': False,
        'rotation_axis': jnp.array([1.0, 0.0, 0.0]),
        'rotation_angle': 0.0,
        'apply_translation': False,
        'translation_vector': jnp.zeros(3),
    }
    return event_simulator(track_params, key, photonsim_data)

simulate_all_particles = jax.vmap(
    simulate_single_particle,
    in_axes=(0, 0, 0, 0, 0, 0, 0, 0, 0),
)

master_key = jax.random.PRNGKey(42)
particle_keys = jax.random.split(master_key, particle_data['n_particles'])

def step_kernel():
    return simulate_all_particles(
        te_d, tp_d, td_d, bo_d, bd_d, bt_d, bw_d, Np_d, particle_keys)

(PE_pp, T_pp), t_kernel = bench(step_kernel, block=True)
print(f'kernel (vmap, n_particles={particle_data["n_particles"]}, '
      f'PAD_SIZE={PAD_SIZE:,}, K={K}):')
print(f'              : {fmt(t_kernel)}')

In [ ]:
# === Step 5 — device -> host (np.asarray on already-materialized buffers) ===
# Make sure the kernel has run before we time the copy.
PE_pp_warm, T_pp_warm = step_kernel()
_block((PE_pp_warm, T_pp_warm))

def step_to_host():
    a = np.asarray(PE_pp_warm, dtype=np.float32)
    b = np.asarray(T_pp_warm,  dtype=np.float32)
    return a, b

_, t_host = bench(step_to_host)
print(f'to_host (asarray on warm buffers): {fmt(t_host)}')

In [ ]:
# === Step 6 — aggregate (jnp.sum / jnp.min) + smearing ===
def step_aggregate_smear():
    PE_true = jnp.sum(PE_pp_warm, axis=0)
    T_true  = jnp.min(jnp.where(T_pp_warm > 0, T_pp_warm, jnp.inf), axis=0)
    T_true  = jnp.where(jnp.isfinite(T_true), T_true, 0.0)
    sk1, sk2 = jax.random.split(jax.random.PRNGKey(0))
    PE_reco = smear_charges_SK_like(PE_true, key=sk1)
    T_reco  = smear_times(T_true, key=sk2)
    return PE_reco, T_reco

_, t_smear = bench(step_aggregate_smear, block=True)
print(f'agg+smear     : {fmt(t_smear)}')

In [ ]:
# === Step 7 — t0 frame shift (pure NumPy where) ===
T_pp_np = np.asarray(T_pp_warm)

def step_t0_shift():
    t0v = np.float32(1.5)
    out = np.where(T_pp_np > 0, T_pp_np + t0v, T_pp_np)
    return out

_, t_t0 = bench(step_t0_shift)
print(f't0_shift      : {fmt(t_t0)}')

In [ ]:
# === Step 8 — build_interaction_metadata ===
def step_meta():
    return build_interaction_metadata(
        particle_data,
        t0=1.5,
        vertex_xyz=np.zeros(3, dtype=np.float32),
        source_type_code=0,  # particle gun
    )

interaction_meta, t_meta = bench(step_meta)
print(f'meta          : {fmt(t_meta)}')

In [ ]:
# === Step 9 — _compute_contained (geometric containment walk) ===
PE_pp_np = np.asarray(PE_pp_warm)

extended_info = {
    'n_particles': particle_data['n_particles'],
    'particles': particle_data['particles'],
    'track_info_dict': particle_data['track_info_dict'],
    'primary_to_interaction': {
        tid: 0 for tid in interaction_meta['primary_track_ids']},
    'interaction_metadata': [interaction_meta],
    'PE_per_particle': PE_pp_np,
    'T_per_particle': T_pp_np,
    'PE_reco': PE_pp_np[0],
    'T_reco':  T_pp_np[0],
    'source':  'PhotonSim_Particles_VMAP',
    'include_track_segments': False,
}

def step_contain():
    return _compute_contained(extended_info, None)

_, t_cont = bench(step_contain)
print(f'contain       : {fmt(t_cont)}')

In [ ]:
# === Summary ===
rows = [
    ('root_read',  t_root),
    ('preprocess', t_pre),
    ('device_put', t_dput),
    ('kernel',     t_kernel),
    ('to_host',    t_host),
    ('agg+smear',  t_smear),
    ('t0_shift',   t_t0),
    ('meta',       t_meta),
    ('contain',    t_cont),
]
total_mean_s = sum(t.mean() for _, t in rows)

print(f'{"stage":<12}  {"mean ms":>9}  {"std ms":>8}  '
      f'{"min ms":>8}  {"max ms":>8}  {"frac":>6}')
print('-' * 60)
for name, t in rows:
    m_ms = 1000 * t.mean()
    print(f'{name:<12}  {m_ms:>9.2f}  {1000*t.std():>8.2f}  '
          f'{1000*t.min():>8.2f}  {1000*t.max():>8.2f}  '
          f'{m_ms/(1000*total_mean_s):>6.1%}')
print('-' * 60)
print(f'{"SUM (1 event)":<12}  {1000*total_mean_s:>9.2f} ms  '
      f'~ {total_mean_s:.2f} s')
print()
print(f'PAD_SIZE: {PAD_SIZE:,}    n_particles in event {EVENT_IDX}: '
      f'{particle_data["n_particles"]}    actual photons: {total_photons:,}')
print(f'kernel  per-event mean: {1000*t_kernel.mean():.2f} ms')
print(f'kernel  per-photon equivalent (over PAD_SIZE x n_particles): '
      f'{1000*t_kernel.mean() / (PAD_SIZE * particle_data["n_particles"]) * 1e6:.3f} ns/photon')

In [ ]:
# === Optional: cross-event sweep ===
# Re-runs the per-event pipeline (root_read + preprocess + device_put +
# kernel + to_host) for every event in the ROOT file. Use this to spot
# events whose actual photon count drives outliers.
def per_event_pipeline(event_idx):
    pd = read_particle_data_from_photonsim(
        ROOT_FILE, event_idx,
        include_track_segments=False, include_segment_index=False)
    n_p = pd['n_particles']
    bo = np.zeros((n_p, PAD_SIZE, 3), dtype=np.float32)
    bd_ = np.tile(default_direction, (n_p, PAD_SIZE, 1))
    bt_ = np.zeros((n_p, PAD_SIZE), dtype=np.float32)
    bw_ = np.zeros((n_p, PAD_SIZE), dtype=np.float32)
    Np_ = np.zeros(n_p, dtype=np.int32)
    te_ = np.zeros(n_p, dtype=np.float32)
    tp_ = np.zeros((n_p, 3), dtype=np.float32)
    td_ = np.zeros((n_p, 3), dtype=np.float32)
    for i, p in enumerate(pd['particles']):
        idx = p['photon_indices']
        N = len(idx); Np_[i] = N
        ti = p['track_info']
        if ti is not None:
            te_[i]=ti['energy']; tp_[i]=ti['position']; td_[i]=ti['direction']
        else:
            te_[i]=pd['primary_energy']; td_[i]=[0.0,0.0,1.0]
        if N > 0:
            bo[i,:N] = pd['photon_origins'][idx].astype(np.float32, copy=False)
            bd_[i,:N] = pd['photon_directions'][idx].astype(np.float32, copy=False)
            bt_[i,:N] = pd['photon_times'][idx].astype(np.float32, copy=False)
            bw_[i,:N] = pd['photon_wavelengths'][idx].astype(np.float32, copy=False)
    bo_j = jax.device_put(bo); bd_j = jax.device_put(bd_)
    bt_j = jax.device_put(bt_); bw_j = jax.device_put(bw_)
    Np_j = jax.device_put(Np_); te_j = jax.device_put(te_)
    tp_j = jax.device_put(tp_); td_j = jax.device_put(td_)
    keys = jax.random.split(jax.random.PRNGKey(event_idx), n_p)
    PE, T = simulate_all_particles(te_j, tp_j, td_j, bo_j, bd_j, bt_j, bw_j, Np_j, keys)
    PE.block_until_ready(); T.block_until_ready()
    return PE.shape

# Warm once to avoid event-0 JIT dominating.
per_event_pipeline(0)

per_event = []
for ei in range(n_events_in_file):
    s = time.perf_counter()
    per_event_pipeline(ei)
    per_event.append(time.perf_counter() - s)
per_event = np.array(per_event)

print(f'per-event pipeline (root+preprocess+device_put+kernel+block):')
print(f'  {fmt(per_event)}')
print()
print('photon counts per event vs per-event time:')
for ei, (n, t) in enumerate(zip(photon_counts, per_event)):
    print(f'  event {ei:2d}: {n:>7,} photons   {1000*t:7.1f} ms')